<a href="https://colab.research.google.com/github/RafaelCaballero/BME/blob/main/mfia/ibexCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IBEX 35: seleccion aleatoria de variables con validacion cruzada

El objetivo es predecir `mapfre_close`, entendido como rendimiento porcentual simple del cierre ajustado de Mapfre, usando variables retardadas del IBEX, incluyendo Mapfre en dias anteriores.


In [5]:
try:
    import yfinance as yf
except ModuleNotFoundError:
    %pip install -q yfinance
    import yfinance as yf

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RepeatedKFold, cross_validate
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


## Valores del IBEX

La lista usa tickers de Yahoo Finance para los componentes del IBEX 35. Los nombres de la izquierda son los prefijos que tendran las columnas del dataframe final.


In [6]:
IBEX_TICKERS = {
    "acs": "ACS.MC",
    "acerinox": "ACX.MC",
    "aena": "AENA.MC",
    "amadeus": "AMS.MC",
    "acciona": "ANA.MC",
    "accionaenergia": "ANE.MC",
    "bbva": "BBVA.MC",
    "bankinter": "BKT.MC",
    "caixabank": "CABK.MC",
    "cellnex": "CLNX.MC",
    "colonial": "COL.MC",
    "endesa": "ELE.MC",
    "enagas": "ENG.MC",
    "fluidra": "FDR.MC",
    "ferrovial": "FER.MC",
    "grifols": "GRF.MC",
    "iag": "IAG.MC",
    "iberdrola": "IBE.MC",
    "indra": "IDR.MC",
    "inditex": "ITX.MC",
    "logista": "LOG.MC",
    "mapfre": "MAP.MC",
    "merlin": "MRL.MC",
    "arcelormittal": "MTS.MC",
    "naturgy": "NTGY.MC",
    "puig": "PUIG.MC",
    "redeia": "RED.MC",
    "repsol": "REP.MC",
    "rovi": "ROVI.MC",
    "sabadell": "SAB.MC",
    "santander": "SAN.MC",
    "sacyr": "SCYR.MC",
    "solaria": "SLR.MC",
    "telefonica": "TEF.MC",
    "unicaja": "UNI.MC",
}


## Descarga y preparacion inicial

La funcion `descarga` baja el cierre ajustado y el volumen. Las columnas `*_close` son rendimientos porcentuales simples calculados como `100 * (precio_t / precio_t-1 - 1)`; las columnas `*_vol` conservan el volumen diario.


In [7]:
def extrae_serie(datos, ticker, campo):
    """Devuelve una serie de yfinance tanto si el MultiIndex viene por ticker como por campo."""
    if not isinstance(datos.columns, pd.MultiIndex):
        return datos[campo]

    nivel_0 = datos.columns.get_level_values(0)
    nivel_1 = datos.columns.get_level_values(1)

    if ticker in nivel_0 and campo in nivel_1:
        return datos[(ticker, campo)]
    if campo in nivel_0 and ticker in nivel_1:
        return datos[(campo, ticker)]

    raise KeyError(f"No encuentro {campo} para {ticker}")


def descarga(fecha_inicial="2021-01-01", tickers=IBEX_TICKERS):
    """Descarga datos de Yahoo Finance y devuelve close pct-ret y volumen del IBEX."""
    lista_yahoo = list(tickers.values())
    datos = yf.download(
        lista_yahoo,
        start=fecha_inicial,
        auto_adjust=False,
        group_by="ticker",
        progress=False,
        threads=True,
    )

    columnas = {}
    avisos = []

    for nombre, ticker in tickers.items():
        try:
            cierre_ajustado = extrae_serie(datos, ticker, "Adj Close")
            volumen = extrae_serie(datos, ticker, "Volume")
        except KeyError as exc:
            avisos.append(str(exc))
            continue

        columnas[f"{nombre}_close"] = cierre_ajustado.pct_change() * 100
        columnas[f"{nombre}_vol"] = volumen

    df = pd.DataFrame(columnas).replace([np.inf, -np.inf], np.nan)
    columnas_vacias = df.columns[df.isna().all()].tolist()
    if columnas_vacias:
        print("Columnas sin datos en Yahoo Finance y eliminadas:")
        print(columnas_vacias)
        df = df.drop(columns=columnas_vacias)

    if avisos:
        print("Avisos de descarga:")
        for aviso in avisos:
            print("-", aviso)

    return df


In [8]:
df = descarga("2021-01-01")
df.head()


,acs_close,acs_vol,acerinox_close,acerinox_vol,aena_close,aena_vol,amadeus_close,amadeus_vol,acciona_close,acciona_vol,...,santander_close,santander_vol,sacyr_close,sacyr_vol,solaria_close,solaria_vol,telefonica_close,telefonica_vol,unicaja_close,unicaja_vol
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-04,NaN,1289994,NaN,708563,NaN,2376120,NaN,832292,NaN,105600,...,NaN,61027452,NaN,6332148,NaN,813240,NaN,13672612,NaN,4865768
2021-01-05,1.629035,449892,0.193684,463928,-0.144413,2422690,0.275568,697035,-0.512375,79442,...,0.350660,34085777,0.050241,2141749,8.401303,1613539,0.727512,10360057,-4.050280,3509214
2021-01-06,2.185798,819258,2.426993,1140915,2.024583,1664940,2.335976,973032,2.575108,162252,...,6.872474,73687945,3.666482,2051218,6.245297,1309795,4.935307,19993932,1.164475,2495990
2021-01-07,-1.105192,2199890,2.956583,1132608,-1.275687,1322150,-4.665999,1298370,5.523015,174305,...,1.471388,54802375,-0.775157,2437088,9.560909,1880786,2.781752,61164685,1.438856,3523667
2021-01-08,0.252336,521614,-1.099804,696155,1.005020,3079400,0.528163,888477,1.427439,158751,...,-1.181551,63402423,-0.878941,1819137,-13.510020,3152745,-0.474323,16315193,-0.070943,3388847


In [10]:
df.to_excel("datos_ibex.xlsx")

In [9]:
df.shape


(1407, 70)

## Construccion de `df_p`

En cada fila dejamos el objetivo `c` del dia actual y, como variables explicativas, todas las columnas retardadas de 1 a `p` dias, incluida `mapfre_close` de dias anteriores. No se usa `mapfre_close` del propio dia como predictor.


In [11]:
def prepara_df_p(df, c="mapfre_close", p=5):
    if c not in df.columns:
        raise ValueError(f"La columna objetivo {c!r} no esta en df")

    objetivo = df[[c]].copy()
    explicativas = df.copy()
    retardos = []

    for lag in range(1, p + 1):
        retardos.append(explicativas.shift(lag).add_suffix(f"_t-{lag}"))

    df_p = pd.concat([objetivo] + retardos, axis=1).dropna()
    return df_p


In [12]:
c = "mapfre_close"
p = 5

df_p = prepara_df_p(df, c=c, p=p)

assert c in df_p.columns
assert any(col.startswith(f"{c}_t-") for col in df_p.columns), "Faltan retardos de Mapfre"

df_p.head()


,mapfre_close,acs_close_t-1,acs_vol_t-1,acerinox_close_t-1,acerinox_vol_t-1,aena_close_t-1,aena_vol_t-1,amadeus_close_t-1,amadeus_vol_t-1,acciona_close_t-1,...,santander_close_t-5,santander_vol_t-5,sacyr_close_t-5,sacyr_vol_t-5,solaria_close_t-5,solaria_vol_t-5,telefonica_close_t-5,telefonica_vol_t-5,unicaja_close_t-5,unicaja_vol_t-5
Date,,,,,,,,,,,,,,,,,,,,,
2024-05-13,0.000000,0.102688,495022.0,1.964632,1046707.0,1.150099,1024880.0,0.289937,702427.0,3.148926,...,0.882908,22839300.0,1.117000,994722.0,0.389105,923717.0,0.328555,9460849.0,-1.179245,5507230.0
2024-05-14,-0.173019,0.923075,403195.0,0.000000,434037.0,1.137010,882330.0,1.445560,761604.0,-1.072598,...,3.446013,48974273.0,1.453481,1801492.0,5.232558,1678033.0,0.163736,14819611.0,1.034203,14582592.0
2024-05-15,1.126527,0.203247,485238.0,0.385362,2349725.0,1.630131,2429340.0,1.836605,1012164.0,1.584653,...,0.920060,37871807.0,2.120347,2426452.0,0.368324,1162240.0,-2.195219,34886935.0,-0.472421,11115668.0
2024-05-16,0.171388,0.101417,736659.0,0.191926,310052.0,0.442473,2388060.0,0.528613,1031916.0,1.724137,...,-0.031452,19080950.0,0.448933,2907789.0,0.917435,910479.0,-0.692462,69466564.0,-0.474697,6608466.0
2024-05-17,0.342167,0.709212,552398.0,1.149431,570369.0,-0.330393,1063780.0,1.546553,839940.0,0.322842,...,-0.744214,20119224.0,0.614520,2633466.0,1.454544,778000.0,-1.178175,29287907.0,1.510320,7420066.0


In [15]:
35*5*2

350

In [13]:
for c in df_p.columns:
    print(c,end=" ")

mapfre_close acs_close_t-1 acs_vol_t-1 acerinox_close_t-1 acerinox_vol_t-1 aena_close_t-1 aena_vol_t-1 amadeus_close_t-1 amadeus_vol_t-1 acciona_close_t-1 acciona_vol_t-1 accionaenergia_close_t-1 accionaenergia_vol_t-1 bbva_close_t-1 bbva_vol_t-1 bankinter_close_t-1 bankinter_vol_t-1 caixabank_close_t-1 caixabank_vol_t-1 cellnex_close_t-1 cellnex_vol_t-1 colonial_close_t-1 colonial_vol_t-1 endesa_close_t-1 endesa_vol_t-1 enagas_close_t-1 enagas_vol_t-1 fluidra_close_t-1 fluidra_vol_t-1 ferrovial_close_t-1 ferrovial_vol_t-1 grifols_close_t-1 grifols_vol_t-1 iag_close_t-1 iag_vol_t-1 iberdrola_close_t-1 iberdrola_vol_t-1 indra_close_t-1 indra_vol_t-1 inditex_close_t-1 inditex_vol_t-1 logista_close_t-1 logista_vol_t-1 mapfre_close_t-1 mapfre_vol_t-1 merlin_close_t-1 merlin_vol_t-1 arcelormittal_close_t-1 arcelormittal_vol_t-1 naturgy_close_t-1 naturgy_vol_t-1 puig_close_t-1 puig_vol_t-1 redeia_close_t-1 redeia_vol_t-1 repsol_close_t-1 repsol_vol_t-1 rovi_close_t-1 rovi_vol_t-1 sabadell_cl

In [ ]:
df_p.shape


## Validacion cruzada

Se sigue el esquema de `19cv.ipynb`: `LinearRegression`, `RepeatedKFold`, `cross_validate`, `mean_squared_error` negativo y conversion posterior a RMSE. Como mezclamos rendimientos y volumenes, la regresion se evalua dentro de un `Pipeline` con `StandardScaler` para escalar las variables predictoras en cada fold.


In [16]:
def LinearRegressionCV(X, y, repeticiones=100, divisiones=10, random_state=None):
    medidor_error = make_scorer(mean_squared_error, greater_is_better=False)
    repartidor = RepeatedKFold(
        n_splits=divisiones,
        n_repeats=repeticiones,
        random_state=random_state,
    )
    metodo = make_pipeline(StandardScaler(), LinearRegression())
    cv_res = cross_validate(
        metodo,
        X,
        y,
        cv=repartidor,
        scoring=medidor_error,
        n_jobs=-1,
    )

    mse_neg = cv_res["test_score"]
    rmse = np.sqrt(-mse_neg)
    return rmse.mean()


## Busqueda aleatoria de columnas

En cada iteracion se eligen `m` columnas al azar entre las variables retardadas y se calcula el RMSE medio con validacion cruzada.


In [ ]:
c = "mapfre_close"
n = 100
m = 12
repeticiones = 50
divisiones = 10
semilla = None

XColumns = [col for col in df_p.columns if col != c]
y = df_p[c]

rng = np.random.default_rng(semilla)
resultados = []

for i in range(n):
    columnas_elegidas = list(rng.choice(XColumns, size=m, replace=False))
    X = df_p[columnas_elegidas]
    rmse_medio = LinearRegressionCV(
        X,
        y,
        repeticiones=repeticiones,
        divisiones=divisiones,
        #random_state=semilla,
    )
    resultados.append({
        "iteracion": i + 1,
        "columnas": columnas_elegidas,
        "rmse": rmse_medio,
    })

    if (i + 1) % 10 == 0:
        print(f"Iteracion {i + 1}/{n}. Mejor RMSE provisional: {min(r['rmse'] for r in resultados):.6f}")

df_res = pd.DataFrame(resultados).sort_values("rmse").reset_index(drop=True)
df_res


Iteracion 10/100. Mejor RMSE provisional: 1.370568
Iteracion 20/100. Mejor RMSE provisional: 1.370568
Iteracion 30/100. Mejor RMSE provisional: 1.364711
Iteracion 40/100. Mejor RMSE provisional: 1.364711


## Estadisticas de los experimentos

Calculamos un pequeno resumen de los `n` experimentos, incluyendo el porcentaje de mejora del mejor RMSE frente al peor.


In [19]:
rmse_mejor = df_res["rmse"].min()
rmse_peor = df_res["rmse"].max()
mejora_pct = 100 * (rmse_peor - rmse_mejor) / rmse_peor

estadisticas_rmse = pd.DataFrame({
    "estadistico": ["mejor", "peor", "media", "mediana", "desviacion_tipica", "mejora_mejor_vs_peor_%"],
    "valor": [
        rmse_mejor,
        rmse_peor,
        df_res["rmse"].mean(),
        df_res["rmse"].median(),
        df_res["rmse"].std(),
        mejora_pct,
    ],
})

print(f"Mejora del mejor RMSE frente al peor: {mejora_pct:.2f}%")
estadisticas_rmse


Mejora del mejor RMSE frente al peor: 4.69%


,estadistico,valor
0,mejor,4.015926e+06
1,peor,4.213460e+06
2,media,4.101319e+06
3,mediana,4.094201e+06
4,desviacion_tipica,5.105977e+04
5,mejora_mejor_vs_peor_%,4.688174e+00


## Mejor combinacion encontrada

In [20]:
mejor = df_res.iloc[0]
mejores_columnas = mejor["columnas"]

print(f"Menor RMSE medio: {mejor['rmse']:.6f}")
print("Columnas seleccionadas:")
for columna in mejores_columnas:
    print("-", columna)


Menor RMSE medio: 4015925.565690
Columnas seleccionadas:
- enagas_vol_t-3
- iberdrola_vol_t-1
- enagas_close_t-1
- acs_close_t-5
- mapfre_close_t-4
- sacyr_vol_t-3
- santander_close_t-4
- amadeus_vol_t-2
- sacyr_vol_t-1
- telefonica_close_t-1
- arcelormittal_vol_t-5
- ferrovial_close_t-4


In [ ]:
mejores_columnas
